Step 1: Import Libraries & Load Prepared Data

1.1 Import Pandas

In [1]:
import pandas as pd

1.2 Load Prepared Dataset

In [2]:
sales_data =pd.read_csv(
    "..//outputs/prepared_sales_data.csv"
)

1.3 Check Dataset Loading

In [3]:
print("Dataset Shape:", sales_data.shape)

Dataset Shape: (1019516, 21)


In [4]:
sales_data.head()

,Order_ID,Product_ID,Product_Name,Customer_ID,Order_Date,Payment_Method,Order_Status,Quantity,Discount,Selling_Price_Details,...,Gross_Sales,Discount_Amount,Net_Sales,Total_Cost,Profit,Profit_Margin,Region,Category_Details,Sub_Category,Brand
0,ORD000001,PRD000116,Britannia Spinach 1L,CUST003379,2023-12-15,Credit Card,Delivered,3,0.00,175,...,525,0.0,525.0,405,120.0,0.228571,South,Fruits & Vegetables,Vegetables,Britannia
1,ORD000001,PRD000344,Sensodyne Orange Juice 200g,CUST003379,2023-12-15,Credit Card,Delivered,1,0.00,50,...,50,0.0,50.0,45,5.0,0.100000,South,Beverages,Juices,Sensodyne
2,ORD000001,PRD000756,Dove Chana Dal Pack of 1,CUST003379,2023-12-15,Credit Card,Delivered,1,0.05,126,...,126,6.3,119.7,105,14.7,0.122807,South,Staples,Pulses,Dove
3,ORD000002,PRD000878,Duracell Chocolate Cookies 1kg,CUST008843,2023-06-06,UPI,Delivered,1,0.15,40,...,40,6.0,34.0,25,9.0,0.264706,West,Bakery,Cookies,Duracell
4,ORD000002,PRD000415,Kissan Potato Chips 200g,CUST008843,2023-06-06,UPI,Delivered,1,0.10,54,...,54,5.4,48.6,35,13.6,0.279835,West,Snacks,Chips,Kissan


1.4 Cheak Importent Columns

In [5]:
sales_data.columns.tolist()

['Order_ID',
 'Product_ID',
 'Product_Name',
 'Customer_ID',
 'Order_Date',
 'Payment_Method',
 'Order_Status',
 'Quantity',
 'Discount',
 'Selling_Price_Details',
 'Cost_Price_Details',
 'Gross_Sales',
 'Discount_Amount',
 'Net_Sales',
 'Total_Cost',
 'Profit',
 'Profit_Margin',
 'Region',
 'Category_Details',
 'Sub_Category',
 'Brand']

Step 2: Create Profit Margin Percentage

In [7]:
sales_data["Profit_Margin"].head()

0    0.228571
1    0.100000
2    0.122807
3    0.264706
4    0.279835
Name: Profit_Margin, dtype: float64

Step 3: Create Order Value Category

3.1 Check Net Sales Distribution

In [8]:
sales_data["Net_Sales"].describe()

count    1.019516e+06
mean     3.744967e+02
std      4.776463e+02
min      7.500000e+00
25%      9.000000e+01
50%      2.090000e+02
75%      4.750000e+02
max      8.640000e+03
Name: Net_Sales, dtype: float64

3.2 Create Order Value Category

In [9]:
import numpy as np

In [11]:
condition = [
    sales_data["Net_Sales"] <  200,
    sales_data["Net_Sales"] <= 500
]
choices = [
    "Low Value",
    "Medium Value"
]

sales_data["Order_Value_Category"] = np.select(
    condition,
    choices,
    default = "High Value"

)

3.3 Verify the New Feature

In [12]:
sales_data[
    ["Net_Sales", "Order_Value_Category"]
].head(10)

,Net_Sales,Order_Value_Category
0,525.00,High Value
1,50.00,Low Value
2,119.70,Low Value
3,34.00,Low Value
4,48.60,Low Value
5,95.00,Low Value
6,28.50,Low Value
7,360.00,Medium Value
8,247.50,Medium Value
9,166.25,Low Value


3.4 Check Category Distribution

In [13]:
sales_data["Order_Value_Category"].value_counts()

Order_Value_Category
Low Value       497418
Medium Value    285284
High Value      236814
Name: count, dtype: int64

In [16]:
conditions = [
    sales_data["Discount"] == 0,
    sales_data["Discount"] <= 0.10,
    sales_data["Discount"] <= 0.20
]

choice = [
    "No Discount",
    "Low Discount",
    "Medium Discount"
]

sales_data["Discount_Category"] = np.select(
    conditions,
    choice,
    default = "High Discount"
)

In [18]:
sales_data[
    ["Discount", "Discount_Category"]
].head(10)

,Discount,Discount_Category
0,0.00,No Discount
1,0.00,No Discount
2,0.05,Low Discount
3,0.15,Medium Discount
4,0.10,Low Discount
5,0.05,Low Discount
6,0.05,Low Discount
7,0.00,No Discount
8,0.10,Low Discount
9,0.05,Low Discount


In [20]:
sales_data["Discount_Category"].value_counts()

Discount_Category
Low Discount       470402
No Discount        362232
Medium Discount    163383
High Discount       23499
Name: count, dtype: int64

Step 5: Create Quantity Category

Check Quantity Distribution

In [21]:
sales_data["Quantity"].value_counts().sort_index()

Quantity
1    562252
2    274283
3    121037
4     46830
5     15114
Name: count, dtype: int64

Create Quantity Category

In [22]:
import numpy as np

conditions = [
    sales_data["Quantity"] <= 2,
    sales_data["Quantity"] <= 5
]

choices = [
    "Low Quantity",
    "Medium Quantity"
]

sales_data["Quantity_Category"] = np.select(
    conditions,
    choices,
    default="High Quantity"
)

Verify

In [23]:
sales_data[
    ["Quantity","Quantity_Category"]
].head(10)

,Quantity,Quantity_Category
0,3,Medium Quantity
1,1,Low Quantity
2,1,Low Quantity
3,1,Low Quantity
4,1,Low Quantity
5,1,Low Quantity
6,1,Low Quantity
7,1,Low Quantity
8,1,Low Quantity
9,1,Low Quantity


In [24]:
sales_data["Quantity_Category"].value_counts()

Quantity_Category
Low Quantity       836535
Medium Quantity    182981
Name: count, dtype: int64

Step 6: Create Customer Purchase Frequency Category

6.1 Calculate Customer Order Frequency

In [31]:
customer_order_frequency =(
    sales_data.groupby("Customer_ID")["Order_ID"]
    .nunique()
    .reset_index(name ="Total_Orders")
)


In [27]:
customer_order_frequency.head()

,Customer_ID,Total_orders
0,CUST000001,19
1,CUST000002,9
2,CUST000003,47
3,CUST000004,8
4,CUST000005,12


6.2 :Create Frequency Category

In [32]:
conditions = [
    customer_order_frequency["Total_Orders"] <= 5,
    customer_order_frequency["Total_Orders"] <= 15
]

choices = [
    "Low Frequency",
    "Medium Frequency"
]

customer_order_frequency["Purchase_Frequency_Category"] = np.select(
    conditions,
    choices,
    default="High Frequency"
)

Verify

In [33]:
customer_order_frequency.head(10)

,Customer_ID,Total_Orders,Purchase_Frequency_Category
0,CUST000001,19,High Frequency
1,CUST000002,9,Medium Frequency
2,CUST000003,47,High Frequency
3,CUST000004,8,Medium Frequency
4,CUST000005,12,Medium Frequency
5,CUST000006,12,Medium Frequency
6,CUST000007,20,High Frequency
7,CUST000008,4,Low Frequency
8,CUST000009,21,High Frequency
9,CUST000010,245,High Frequency


In [37]:
customer_order_frequency[
    "Purchase_Frequency_Category"
].value_counts()

Purchase_Frequency_Category
High Frequency      6027
Medium Frequency    2597
Low Frequency       1267
Name: count, dtype: int64

Step 7: Merge Customer Purchase Frequency with Sales Data

Merge with Sales Data

In [39]:
sales_data = sales_data.merge(
    customer_order_frequency[
        ["Customer_ID", "Total_Orders", "Purchase_Frequency_Category"]
    ],
    on="Customer_ID",
    how="left"
)

Verify the New Columns

In [41]:
sales_data[
    [
        "Customer_ID",
        "Total_Orders",
        "Purchase_Frequency_Category"
    ]

].head(10)

,Customer_ID,Total_Orders,Purchase_Frequency_Category
0,CUST003379,188,High Frequency
1,CUST003379,188,High Frequency
2,CUST003379,188,High Frequency
3,CUST008843,40,High Frequency
4,CUST008843,40,High Frequency
5,CUST008484,64,High Frequency
6,CUST000833,20,High Frequency
7,CUST009978,75,High Frequency
8,CUST009978,75,High Frequency
9,CUST006677,173,High Frequency


Cheak Missing Values

In [42]:
sales_data[
    ["Total_Orders","Purchase_Frequency_Category"]
].isna().sum()

Total_Orders                   0
Purchase_Frequency_Category    0
dtype: int64

Cheak Dataset Shape

In [43]:
print("Dataset Shape:", sales_data.shape)

Dataset Shape: (1019516, 26)


Step 8: Create Customer Spending Category.

8.1 Calculate Customer Total Spending

In [46]:
customer_spending = (
    sales_data.groupby("Customer_ID")["Net_Sales"]
    .sum()
    .reset_index(name = "Total_Spending")
)
customer_spending.head()

,Customer_ID,Total_Spending
0,CUST000001,16056.00
1,CUST000002,8770.35
2,CUST000003,42151.75
3,CUST000004,4619.20
4,CUST000005,10003.30


Create Spending Category

In [47]:
condition  = [
    customer_spending["Total_Spending"] < 1000,
    customer_spending["Total_Spending"] <= 5000
]

choices = [
    "Low Spender",
    "Medium Spender"
]

customer_spending["Spending_category"] = np.select(
    conditions,
    choices,
    default = "High Spender"
)

verify

In [48]:
customer_spending.head(10)

,Customer_ID,Total_Spending,Spending_category
0,CUST000001,16056.00,High Spender
1,CUST000002,8770.35,Medium Spender
2,CUST000003,42151.75,High Spender
3,CUST000004,4619.20,Medium Spender
4,CUST000005,10003.30,Medium Spender
5,CUST000006,11562.05,Medium Spender
6,CUST000007,16939.10,High Spender
7,CUST000008,4034.80,Low Spender
8,CUST000009,19923.00,High Spender
9,CUST000010,227307.05,High Spender


In [49]:
customer_spending["Spending_category"].value_counts()

Spending_category
High Spender      6027
Medium Spender    2597
Low Spender       1267
Name: count, dtype: int64

Step 9: Merge Customer Spending Category with Sales Data

 Merge Customer Spending Data

In [50]:
sales_data = sales_data.merge(
    customer_spending[
        ["Customer_ID","Total_Spending","Spending_category"]
    ],
    on="Customer_ID",
    how ="left"
)

Verify New Columns

In [51]:
sales_data[
    [
        "Customer_ID",
        "Total_Spending",
        "Spending_category"
    ]
].head(10)

,Customer_ID,Total_Spending,Spending_category
0,CUST003379,171528.40,High Spender
1,CUST003379,171528.40,High Spender
2,CUST003379,171528.40,High Spender
3,CUST008843,33898.05,High Spender
4,CUST008843,33898.05,High Spender
5,CUST008484,53587.15,High Spender
6,CUST000833,14413.75,High Spender
7,CUST009978,83615.35,High Spender
8,CUST009978,83615.35,High Spender
9,CUST006677,157423.85,High Spender


Cheak Missing Value

In [52]:
sales_data[
    ["Total_Spending","Spending_category"]

].isna().sum()

Total_Spending       0
Spending_category    0
dtype: int64

Cheak Dataset Shape

In [53]:
print("Dateset Shapes",sales_data.shape)

Dateset Shapes (1019516, 28)


Step 10: Create Customer Segment

Create Customer Segment

In [55]:
conditions = [
    (sales_data["Purchase_Frequency_Category"] == "High Frequency") &
    (sales_data["Spending_category"] == "High Spender"),

    (sales_data["Purchase_Frequency_Category"] == "High Frequency"),

    (sales_data["Spending_category"] == "High spender"),

    (sales_data["Purchase_Frequency_Category"] == "Low Frequency") &
    (sales_data["Spending_category"] == "Low Spender")
]

choices = [
    "Loyal High-Value customer",
    "Frequent Customer",
    "High-value Customer",
    "Low-value Customer"
]

sales_data["Customer_Segment"] = np.select(
    conditions,
    choices,
    default="Regular Customer"
)

Verify

In [57]:
sales_data[
    [
        "Customer_ID",
        "Purchase_Frequency_Category",
        "Spending_category",
        "Customer_Segment"
    ]
].head(10)

,Customer_ID,Purchase_Frequency_Category,Spending_category,Customer_Segment
0,CUST003379,High Frequency,High Spender,Loyal High-Value customer
1,CUST003379,High Frequency,High Spender,Loyal High-Value customer
2,CUST003379,High Frequency,High Spender,Loyal High-Value customer
3,CUST008843,High Frequency,High Spender,Loyal High-Value customer
4,CUST008843,High Frequency,High Spender,Loyal High-Value customer
5,CUST008484,High Frequency,High Spender,Loyal High-Value customer
6,CUST000833,High Frequency,High Spender,Loyal High-Value customer
7,CUST009978,High Frequency,High Spender,Loyal High-Value customer
8,CUST009978,High Frequency,High Spender,Loyal High-Value customer
9,CUST006677,High Frequency,High Spender,Loyal High-Value customer


Check Segment Distribution

In [59]:
sales_data["Customer_Segment"].value_counts()

Customer_Segment
Loyal High-Value customer    941339
Regular Customer              67637
Low-value Customer            10540
Name: count, dtype: int64

Step 11: Create Profit Category

11.1:Check Profit Distribution

In [60]:
sales_data["Profit"].describe()

count    1.019516e+06
mean     6.270878e+01
std      1.035539e+02
min     -1.142000e+03
25%      1.000000e+01
50%      2.950000e+01
75%      7.500000e+01
max      2.664000e+03
Name: Profit, dtype: float64

11.2: Create Profit category

In [61]:
conditions = [
    sales_data["Profit"] < 100,
    sales_data["Profit"] <= 300
]

choices = [
    "Low Profit",
    "Medium Profit"
]

sales_data["Profit_Category"] = np.select(
    conditions,
    choices,
    default="High Profit"
)

Verify

In [62]:
sales_data[
    ["Profit", "Profit_Category"]
].head(10)

,Profit,Profit_Category
0,120.00,Medium Profit
1,5.00,Low Profit
2,14.70,Low Profit
3,9.00,Low Profit
4,13.60,Low Profit
5,10.00,Low Profit
6,8.50,Low Profit
7,65.00,Low Profit
8,27.50,Low Profit
9,51.25,Low Profit


In [63]:
sales_data["Profit_Category"].value_counts()

Profit_Category
Low Profit       827164
Medium Profit    157071
High Profit       35281
Name: count, dtype: int64

Step 12: Final Feature Engineering Verification

In [67]:
new_features = [
    "Order_Value_Category",
    "Discount_Category",
    "Quantity_Category",
    "Total_Orders",
    "Purchase_Frequency_Category",
    "Total_Spending",
    "Spending_category",
    "Customer_Segment",
    "Profit_Category"
]

sales_data[new_features].head(10)

,Order_Value_Category,Discount_Category,Quantity_Category,Total_Orders,Purchase_Frequency_Category,Total_Spending,Spending_category,Customer_Segment,Profit_Category
0,High Value,No Discount,Medium Quantity,188,High Frequency,171528.40,High Spender,Loyal High-Value customer,Medium Profit
1,Low Value,No Discount,Low Quantity,188,High Frequency,171528.40,High Spender,Loyal High-Value customer,Low Profit
2,Low Value,Low Discount,Low Quantity,188,High Frequency,171528.40,High Spender,Loyal High-Value customer,Low Profit
3,Low Value,Medium Discount,Low Quantity,40,High Frequency,33898.05,High Spender,Loyal High-Value customer,Low Profit
4,Low Value,Low Discount,Low Quantity,40,High Frequency,33898.05,High Spender,Loyal High-Value customer,Low Profit
5,Low Value,Low Discount,Low Quantity,64,High Frequency,53587.15,High Spender,Loyal High-Value customer,Low Profit
6,Low Value,Low Discount,Low Quantity,20,High Frequency,14413.75,High Spender,Loyal High-Value customer,Low Profit
7,Medium Value,No Discount,Low Quantity,75,High Frequency,83615.35,High Spender,Loyal High-Value customer,Low Profit
8,Medium Value,Low Discount,Low Quantity,75,High Frequency,83615.35,High Spender,Loyal High-Value customer,Low Profit
9,Low Value,Low Discount,Low Quantity,173,High Frequency,157423.85,High Spender,Loyal High-Value customer,Low Profit


Check Missing Values

In [68]:
sales_data[new_features].isna().sum()

Order_Value_Category           0
Discount_Category              0
Quantity_Category              0
Total_Orders                   0
Purchase_Frequency_Category    0
Total_Spending                 0
Spending_category              0
Customer_Segment               0
Profit_Category                0
dtype: int64

Cheak Dataset Shape

In [69]:
print("final Dataset Shape:", sales_data.shape)

final Dataset Shape: (1019516, 30)


Save Feature-Engineered Dataset

In [70]:
sales_data.to_csv(
    "../outputs/feature_engineered_sales_data.csv",
    index=False
)